Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

In [ ]:
NAME = ""
COLLABORATORS = ""

---

# Lab 7: Efficient Frontier - Alternative Approach (Solution)

## Introduction

In class, you learned **Method 1** for constructing the efficient frontier: for each target return level, find the portfolio with minimum volatility. This is the classical Markowitz approach.

In this lab, you will implement **Method 2**: for each target volatility level, find the portfolio with maximum return. While the approach is different, both methods trace out the same efficient frontier.

This lab demonstrates the duality principle in optimization theory: minimizing risk for a given return is equivalent to maximizing return for a given risk.

## Mathematical Formulation

### Method 1 (from class): Fixed Return Approach

**Objective:** Minimize portfolio volatility
$$\min \sigma_p = \sqrt{\mathbf{w}^T \mathbf{\Sigma} \mathbf{w}}$$

**Subject to:**
- Target return constraint: $\mathbf{w}^T \mathbf{E}(R) = R_{target}$
- Budget constraint: $\sum_{i=1}^{n} w_i = 1$
- No short-selling: $w_i \geq 0$ for all $i$

### Method 2 (this lab): Fixed Volatility Approach

**Objective:** Maximize portfolio return
$$\max E(R_p) = \mathbf{w}^T \mathbf{E}(R)$$

**Subject to:**
- Target volatility constraint: $\sqrt{\mathbf{w}^T \mathbf{\Sigma} \mathbf{w}} = \sigma_{target}$
- Budget constraint: $\sum_{i=1}^{n} w_i = 1$
- No short-selling: $w_i \geq 0$ for all $i$

**Key Insight:** Both methods identify the same efficient frontier due to the duality principle in optimization.

## Setup and Data Loading

In [ ]:
import pandas as pd  
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.optimize as sco
import os

np.random.seed(777)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
# Load stock prices from CSV data in Risk folder
stock_prices_file = 'stock_prices.csv'

if os.path.exists(stock_prices_file):
    # Load real market data
    print("🔄 Loading stock price data from CSV...")
    df_prices = pd.read_csv('stock_prices.csv', index_col=0, parse_dates=True)
    
    # Select stocks from different sectors for better diversification:
    # AAPL - Technology (Apple)
    # JPM - Financials (JPMorgan Chase)
    # JNJ - Healthcare (Johnson & Johnson)
    # WMT - Consumer Staples (Walmart)
    stocks = ['AAPL', 'JPM', 'JNJ', 'WMT']
    sector_labels = ['Technology', 'Financials', 'Healthcare', 'Consumer Staples']
    
    # Check which stocks are available
    available_stocks = [stock for stock in stocks if stock in df_prices.columns]
    
    if len(available_stocks) < 3:
        print(f"⚠️  Only {len(available_stocks)} stocks available. Need at least 3.")
        print(f"Available: {available_stocks}")
        raise Exception("Insufficient stock data")
    
    # Use available stocks
    df_prices = df_prices[available_stocks]
    
    # Filter date range (use available data range)
    # Keep approximately 1-2 years of data for consistency
    if len(df_prices) > 504:  # If we have more than 2 years
        df_prices = df_prices.iloc[-504:]  # Use last 2 years
    
    print(f"✅ Loaded {len(available_stocks)} stocks from different sectors:")
    for stock, sector in zip(available_stocks, sector_labels[:len(available_stocks)]):
        print(f"   • {stock} - {sector}")
    print(f"📅 Date range: {df_prices.index[0].date()} to {df_prices.index[-1].date()}")
    print(f"📊 Total observations: {len(df_prices)}")
    
else:
    print("⚠️  stock_prices.csv not found in Risk folder")
    raise FileNotFoundError("Please ensure stock_prices.csv exists in the Risk folder")

In [ ]:
# Create table with closing prices
table = df_prices.copy()

# Calculate returns and statistics
returns = table.pct_change()
mean_returns = returns.mean()
cov_matrix = returns.cov()
risk_free_rate = 0.0178

In [ ]:
print("Mean Daily Returns:")
print(mean_returns)
print("\nCovariance Matrix:")
print(cov_matrix)

## Helper Functions

In [ ]:
def portfolio_annualised_performance(weights, mean_returns, cov_matrix):
    """
    Calculate annualized return and volatility for a portfolio.
    
    Parameters:
    -----------
    weights : array-like
        Portfolio weights
    mean_returns : array-like
        Mean daily returns for each asset
    cov_matrix : array-like
        Covariance matrix of daily returns
    
    Returns:
    --------
    tuple : (annualized_volatility, annualized_return)
    """
    returns = np.sum(mean_returns * weights) * 252
    std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(252)
    return std, returns

## Method 2: Fixed Volatility Approach

In this section, we implement the alternative approach to finding the efficient frontier. Instead of minimizing risk for a given return, we maximize return for a given risk level.

### Objective Function: Negative Portfolio Return

Since scipy's `minimize` function only minimizes (there's no maximize), we need to minimize the **negative** of the portfolio return. This is mathematically equivalent to maximizing the return.

$$\text{minimize} \quad -E(R_p) = -\mathbf{w}^T \mathbf{E}(R)$$

In [ ]:
def portfolio_neg_return(weights, mean_returns, cov_matrix):
    """
    Calculate negative portfolio return (for minimization).
    
    We want to MAXIMIZE return, but scipy only has minimize(),
    so we minimize the NEGATIVE return instead.
    
    Parameters:
    -----------
    weights : array-like
        Portfolio weights
    mean_returns : array-like
        Mean daily returns for each asset
    cov_matrix : array-like
        Covariance matrix (not used here, but needed for consistency)
    
    Returns:
    --------
    float : negative annualized return
    """
    return -portfolio_annualised_performance(weights, mean_returns, cov_matrix)[1]

### Constraints for Method 2

We need to set up two types of constraints:

1. **Volatility Constraint**: The portfolio volatility must equal the target volatility
   - Type: 'eq' (equality constraint)
   - Function: $\sqrt{\mathbf{w}^T \mathbf{\Sigma} \mathbf{w}} \times \sqrt{252} = \sigma_{target}$
   - In constraint form: $\sqrt{\mathbf{w}^T \mathbf{\Sigma} \mathbf{w}} \times \sqrt{252} - \sigma_{target} = 0$

2. **Budget Constraint**: All weights must sum to 1
   - Type: 'eq' (equality constraint)
   - Function: $\sum_{i=1}^{n} w_i = 1$
   - In constraint form: $\sum_{i=1}^{n} w_i - 1 = 0$

**Understanding Constraint Syntax:**
The constraint `{'type': 'eq', 'fun': lambda x: np.sum(x) - 1}` means:
- Move everything to one side of the equation
- `np.sum(x) == 1` becomes `np.sum(x) - 1 == 0`
- The function returns what should equal zero

### Bounds for Optimization

Bounds constrain each individual weight:
- Each weight $w_i$ must be between 0 and 1
- This enforces **no short-selling** (weights cannot be negative)
- This also prevents any single asset from exceeding 100% allocation

$$0 \leq w_i \leq 1 \quad \text{for all } i$$

In [ ]:
def efficient_vol(mean_returns, cov_matrix, target_vol):
    """
    Find the portfolio with maximum return for a given target volatility.
    
    This is Method 2: instead of minimizing risk for a target return,
    we maximize return for a target risk.
    
    Parameters:
    -----------
    mean_returns : array-like
        Mean daily returns for each asset
    cov_matrix : array-like
        Covariance matrix of daily returns
    target_vol : float
        Target annualized volatility
    
    Returns:
    --------
    OptimizeResult : scipy optimization result containing optimal weights
    """
    num_assets = len(mean_returns)
    args = (mean_returns, cov_matrix)
    
    # Helper function to calculate portfolio volatility
    def portfolio_volatility(weights):
        return portfolio_annualised_performance(weights, mean_returns, cov_matrix)[0]
    
    # Constraint 1: Portfolio volatility must equal target volatility
    # Constraint 2: Weights must sum to 1
    constraints = (
        {'type': 'eq', 'fun': lambda x: portfolio_volatility(x) - target_vol},
        {'type': 'eq', 'fun': lambda x: np.sum(x) - 1}
    )
    
    # Bounds: each weight between 0 and 1 (no short-selling)
    bounds = tuple((0, 1) for asset in range(num_assets))
    
    # Initial guess: equal weights
    initial_guess = num_assets * [1. / num_assets]
    
    # Minimize negative return (i.e., maximize return)
    result = sco.minimize(
        portfolio_neg_return,
        initial_guess,
        args=args,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )
    
    return result

### Generate the Efficient Frontier

To construct the complete efficient frontier, we:
1. Define a range of target volatility levels
2. For each volatility level, find the portfolio with maximum return
3. Collect all these optimal portfolios to trace out the frontier

**Important:** We need to filter the results to keep only the **upward-sloping** portion of the curve. Portfolios on the downward-sloping part are inefficient (dominated by portfolios with less risk and higher return).

In [ ]:
def efficient_frontier_fixed_vol(mean_returns, cov_matrix, vol_range):
    """
    Generate the efficient frontier using the fixed volatility approach.
    
    For each target volatility in the range, find the portfolio with
    maximum return. This traces out the efficient frontier.
    
    Parameters:
    -----------
    mean_returns : array-like
        Mean daily returns for each asset
    cov_matrix : array-like
        Covariance matrix of daily returns
    vol_range : array-like
        Array of target volatility levels
    
    Returns:
    --------
    list : list of optimization results for each volatility level
    """
    efficients = []
    for target_vol in vol_range:
        try:
            result = efficient_vol(mean_returns, cov_matrix, target_vol)
            if result['success']:
                efficients.append(result)
        except:
            # Skip volatility levels where optimization fails
            pass
    return efficients

### Visualization Function

This function will:
1. Calculate the maximum Sharpe ratio portfolio
2. Calculate the minimum volatility portfolio
3. Generate the efficient frontier using Method 2
4. Plot all results including individual stocks for comparison

In [ ]:
def display_ef_with_selected(mean_returns, cov_matrix, risk_free_rate):
    """
    Display the efficient frontier with optimal portfolios and individual stocks.
    
    This function uses Method 2 (fixed volatility approach) to generate the
    efficient frontier.
    
    Parameters:
    -----------
    mean_returns : array-like
        Mean daily returns for each asset
    cov_matrix : array-like
        Covariance matrix of daily returns
    risk_free_rate : float
        Annual risk-free rate
    """
    
    # Helper functions for max Sharpe and min volatility
    def neg_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate):
        p_var, p_ret = portfolio_annualised_performance(weights, mean_returns, cov_matrix)
        return -(p_ret - risk_free_rate) / p_var
    
    def portfolio_volatility(weights, mean_returns, cov_matrix):
        return portfolio_annualised_performance(weights, mean_returns, cov_matrix)[0]
    
    def max_sharpe_ratio(mean_returns, cov_matrix, risk_free_rate):
        num_assets = len(mean_returns)
        args = (mean_returns, cov_matrix, risk_free_rate)
        constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
        bound = (0.0, 1.0)
        bounds = tuple(bound for asset in range(num_assets))
        result = sco.minimize(neg_sharpe_ratio, num_assets*[1./num_assets,], args=args,
                            method='SLSQP', bounds=bounds, constraints=constraints)
        return result
    
    def min_variance(mean_returns, cov_matrix):
        num_assets = len(mean_returns)
        args = (mean_returns, cov_matrix)
        constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
        bound = (0.0, 1.0)
        bounds = tuple(bound for asset in range(num_assets))
        result = sco.minimize(portfolio_volatility, num_assets*[1./num_assets,], args=args,
                            method='SLSQP', bounds=bounds, constraints=constraints)
        return result
    
    # Calculate optimal portfolios
    max_sharpe = max_sharpe_ratio(mean_returns, cov_matrix, risk_free_rate)
    sdp, rp = portfolio_annualised_performance(max_sharpe['x'], mean_returns, cov_matrix)
    max_sharpe_allocation = pd.DataFrame(max_sharpe.x, index=table.columns, columns=['allocation'])
    max_sharpe_allocation.allocation = [round(i*100, 2) for i in max_sharpe_allocation.allocation]
    max_sharpe_allocation = max_sharpe_allocation.T
    
    min_vol = min_variance(mean_returns, cov_matrix)
    sdp_min, rp_min = portfolio_annualised_performance(min_vol['x'], mean_returns, cov_matrix)
    min_vol_allocation = pd.DataFrame(min_vol.x, index=table.columns, columns=['allocation'])
    min_vol_allocation.allocation = [round(i*100, 2) for i in min_vol_allocation.allocation]
    min_vol_allocation = min_vol_allocation.T
    
    # Calculate individual stock statistics
    an_vol = np.std(returns) * np.sqrt(252)
    an_rt = mean_returns * 252
    
    # Print portfolio allocations
    print("-" * 80)
    print("Maximum Sharpe Ratio Portfolio Allocation\n")
    print("Annualised Return:", round(rp, 2))
    print("Annualised Volatility:", round(sdp, 2))
    print("\n")
    print(max_sharpe_allocation)
    print("-" * 80)
    print("Minimum Volatility Portfolio Allocation\n")
    print("Annualised Return:", round(rp_min, 2))
    print("Annualised Volatility:", round(sdp_min, 2))
    print("\n")
    print(min_vol_allocation)
    print("-" * 80)
    print("Individual Stock Returns and Volatility\n")
    for i, txt in enumerate(table.columns):
        print(txt, ":", "annualised return", round(an_rt[i], 2), 
              ", annualised volatility:", round(an_vol[i], 2))
    print("-" * 80)
    
    # Generate efficient frontier using Method 2 (fixed volatility)
    # Create a range of target volatilities from min_vol to a reasonable maximum
    vol_range = np.linspace(sdp_min, sdp_min * 2.5, 50)
    efficient_portfolios = efficient_frontier_fixed_vol(mean_returns, cov_matrix, vol_range)
    
    # Extract volatilities and returns from efficient portfolios
    ef_vols = []
    ef_rets = []
    for p in efficient_portfolios:
        vol, ret = portfolio_annualised_performance(p['x'], mean_returns, cov_matrix)
        ef_vols.append(vol)
        ef_rets.append(ret)
    
    # Filter to keep only upward-sloping (efficient) part
    # Sort by volatility and keep only points where return increases with volatility
    if len(ef_vols) > 0:
        sorted_indices = np.argsort(ef_vols)
        ef_vols_sorted = [ef_vols[i] for i in sorted_indices]
        ef_rets_sorted = [ef_rets[i] for i in sorted_indices]
        
        # Keep only upward-sloping portion
        filtered_vols = [ef_vols_sorted[0]]
        filtered_rets = [ef_rets_sorted[0]]
        for i in range(1, len(ef_vols_sorted)):
            if ef_rets_sorted[i] > filtered_rets[-1]:
                filtered_vols.append(ef_vols_sorted[i])
                filtered_rets.append(ef_rets_sorted[i])
    else:
        filtered_vols = []
        filtered_rets = []
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 7))
    
    # Plot individual stocks
    ax.scatter(an_vol, an_rt, marker='o', s=200, alpha=0.6, label='Individual Stocks')
    for i, txt in enumerate(table.columns):
        ax.annotate(txt, (an_vol[i], an_rt[i]), xytext=(10, 0), textcoords='offset points')
    
    # Plot optimal portfolios
    ax.scatter(sdp, rp, marker='*', color='r', s=500, label='Maximum Sharpe Ratio', zorder=5)
    ax.scatter(sdp_min, rp_min, marker='*', color='g', s=500, label='Minimum Volatility', zorder=5)
    
    # Plot efficient frontier (Method 2)
    if len(filtered_vols) > 0:
        ax.plot(filtered_vols, filtered_rets, linestyle='-.', color='black', 
                linewidth=2, label='Efficient Frontier (Method 2)', zorder=4)
    
    ax.set_title('Efficient Frontier using Fixed Volatility Approach (Method 2)', fontsize=14)
    ax.set_xlabel('Annualised Volatility', fontsize=12)
    ax.set_ylabel('Annualised Returns', fontsize=12)
    ax.legend(labelspacing=0.8, fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Execute and Plot

In [ ]:
# Generate and display the efficient frontier using Method 2
display_ef_with_selected(mean_returns, cov_matrix, risk_free_rate)

## Discussion Questions

### 1. Visual Comparison

**Question:** Compare your efficient frontier (fixed volatility approach) with the one from class (fixed return approach). Do they look the same?

**Answer:**

Yes, the two efficient frontiers should look identical. Both methods trace out the same curve in risk-return space. This is because:

- Both approaches identify the same set of **Pareto optimal** portfolios
- The efficient frontier represents all portfolios where you cannot improve return without increasing risk
- Method 1 (from class) walks along the frontier by fixing return levels
- Method 2 (this lab) walks along the frontier by fixing volatility levels
- They arrive at the same points, just parameterized differently

Visually, if you overlay both plots, the curves should overlap almost perfectly. Any small differences are due to numerical optimization precision, not fundamental mathematical differences.

**Key Insight:** The efficient frontier is a unique mathematical object. No matter which optimization approach you use to find it, you will always discover the same set of efficient portfolios.

### 2. Mathematical Formulation

**Question:** Write down both optimization problems and explain why they are equivalent.

**Traditional Approach (Method 1 - from class):**

**Minimize:** Portfolio volatility 
$$\sigma_p = \sqrt{\mathbf{w}^T \mathbf{\Sigma} \mathbf{w}}$$

**Subject to:**
- $\mathbf{w}^T \mathbf{E}(R) = R_{target}$ (achieve target return)
- $\sum_{i=1}^{n} w_i = 1$ (weights sum to 1)
- $w_i \geq 0$ for all $i$ (no short selling)

---

**Alternative Approach (Method 2 - this lab):**

**Maximize:** Portfolio return 
$$R_p = \mathbf{w}^T \mathbf{E}(R)$$

**Subject to:**
- $\sqrt{\mathbf{w}^T \mathbf{\Sigma} \mathbf{w}} = \sigma_{target}$ (achieve target volatility)
- $\sum_{i=1}^{n} w_i = 1$ (weights sum to 1)
- $w_i \geq 0$ for all $i$ (no short selling)

---

**Why are they equivalent?**

These two formulations are **dual** problems in optimization theory. The duality principle states that:

1. **Mathematical Duality**: Every constrained optimization problem has a dual formulation. In portfolio optimization:
   - Minimizing risk for a given return (primal problem)
   - Is equivalent to maximizing return for a given risk (dual problem)

2. **Same Efficient Set**: Both methods identify the exact same efficient frontier because:
   - They both enforce the Pareto optimality condition
   - Any portfolio on the frontier cannot be improved in one dimension without worsening the other
   - This optimality condition is independent of which variable we constrain vs. optimize

3. **Different Parameterization**: 
   - Method 1 parameterizes the frontier by return levels: "For each $R_{target}$, find minimum $\sigma_p$"
   - Method 2 parameterizes the frontier by risk levels: "For each $\sigma_{target}$, find maximum $R_p$"
   - Both trace the same curve, just walking along it differently

4. **Fundamental Result in MPT**: This equivalence is a cornerstone of Modern Portfolio Theory. Harry Markowitz's groundbreaking insight was that the efficient frontier is unique regardless of the computational approach used to find it.

**Practical Implication**: 
You can choose whichever method is more convenient for your application:
- Use Method 1 if you have a target return in mind
- Use Method 2 if you have a risk budget or volatility constraint

Both will lead you to the same optimal portfolio allocation.

## Summary

In this lab, you implemented an alternative approach to constructing the efficient frontier:

1. **Method 2**: Fixed volatility approach
   - For each target volatility, maximize return
   - Demonstrates the duality principle in optimization

2. **Key Learning**: Both methods produce identical efficient frontiers
   - Same optimal portfolios, different computational paths
   - Validates the uniqueness of the efficient frontier in MPT

3. **Practical Application**: Choose the method based on your constraints
   - Risk-constrained investors: Use Method 2
   - Return-targeted investors: Use Method 1

This equivalence is a fundamental result in portfolio theory and demonstrates the elegance of Markowitz's Modern Portfolio Theory framework.